# Adapter Smoke Test

Directly load a trained continual SD-LoRA adapter on top of the DINO backbone, inspect its metadata, run one prediction, and optionally sanity-check a small folder of images.

This notebook now scans the project folder and common Drive roots for adapter bundles, then lets you choose which one to load.

Manual path inputs are accepted too:
- `ADAPTER_DIR`: adapter folder, parent export folder, telemetry run folder, telemetry `artifacts/` folder, or `adapter_meta.json`
- `ADAPTER_ROOT`: parent of crop folders such as `models/adapters/`
- `IMAGE_PATH`: one image file
- `BATCH_IMAGE_DIR`: one folder containing image files

Retry controls for the same session:
- set `FORCE_ADAPTER_RESCAN = True` and rerun the adapter discovery cell to rebuild the list
- rerunning the single-image cell now requests a fresh upload by default
- set `REUSE_EXISTING_IMAGE_PATH = True` before rerunning the single-image cell when you intentionally want to keep the previous upload
- set `FORCE_IMAGE_UPLOAD = True` to clear any reused path and force a new upload immediately

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/drive/MyDrive/bitirme projesi'),
    Path('/content/drive/MyDrive/bitirmeprojesi'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _mount_drive_inline() -> None:
    if not _running_in_colab_inline():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Drive mount skipped during bootstrap: {exc}')

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    _mount_drive_inline()
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repository bootstrap failed. Set AADS_REPO_ROOT to an existing checkout, or provide '
        'GH_TOKEN/GITHUB_TOKEN for private GitHub auto-clone.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    mount_drive_if_available()
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)


In [ ]:
import json
from collections import Counter
from pathlib import Path

from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from scripts.colab_adapter_smoke_test import (
    discover_adapter_candidates,
    load_adapter_summary,
    predict_image_folder,
    predict_single_image,
)

# Direct smoke-test controls.
CROP_NAME = None
# Optional crop filter, e.g. 'tomato'. Leave as None to show all discovered adapters.

ADAPTER_DIR = None
# Optional manual override. Accepted values include:
# - .../continual_sd_lora_adapter/
# - .../artifacts/adapter/
# - outputs/colab_notebook_training/
# - .../telemetry/<RUN_ID>/ or .../telemetry/<RUN_ID>/artifacts/
# - .../adapter_meta.json

ADAPTER_ROOT = None
# Fallback deployed root. This should normally be the parent of crop folders, e.g. models/adapters/.

SEARCH_ROOTS = [
    ROOT,
    ROOT / 'outputs',
    ROOT / 'models',
    ROOT / 'models' / 'adapters',
    Path('/content/drive/MyDrive/aads_ulora'),
]
SELECTED_ADAPTER_INDEX = 0
FORCE_ADAPTER_RESCAN = False
IMAGE_PATH = None
FORCE_IMAGE_UPLOAD = False
# One image file path such as /content/sample_leaf.jpg or D:/data/sample_leaf.png
BATCH_IMAGE_DIR = None
# One folder path containing image files for the optional batch sanity check
DEVICE = 'cuda'
CONFIG_ENV = 'colab'
REUSE_EXISTING_IMAGE_PATH = False

print(f'device={DEVICE} config_env={CONFIG_ENV}')

print('Manual ADAPTER_DIR overrides discovery. Otherwise the notebook scans SEARCH_ROOTS and lists every adapter it finds.')
print('Discovery roots:')
for root in SEARCH_ROOTS:
    print(f'- {root}')
print('ADAPTER_ROOT should be the parent of crop folders, IMAGE_PATH should be one file, and BATCH_IMAGE_DIR should be one folder.')
print('Set FORCE_ADAPTER_RESCAN=True to rebuild the adapter list.')
print('The single-image cell now requests a fresh upload by default on rerun. Set REUSE_EXISTING_IMAGE_PATH=True if you want to keep the current IMAGE_PATH, or FORCE_IMAGE_UPLOAD=True to force a new upload.')

In [ ]:
adapter_candidates = []
adapter_selector = None

if FORCE_ADAPTER_RESCAN:
    ADAPTER_DIR = None
    print('FORCE_ADAPTER_RESCAN=True, rebuilding adapter discovery list.')

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Using manual ADAPTER_DIR: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'No adapters found under SEARCH_ROOTS. Set ADAPTER_DIR manually to an adapter folder, telemetry run folder, export folder, or adapter_meta.json, or update SEARCH_ROOTS.'
        )

    print('Discovered adapters:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Choose an adapter in the dropdown, then run the next cell.')
    else:
        print('ipywidgets unavailable. Set SELECTED_ADAPTER_INDEX to the adapter you want, then run the next cell.')

In [ ]:
if ADAPTER_DIR is None:
    selected_index = adapter_selector.value if adapter_selector is not None else SELECTED_ADAPTER_INDEX
    selected_candidate = adapter_candidates[int(selected_index)]
    ADAPTER_DIR = selected_candidate['adapter_dir']
    if CROP_NAME is None:
        CROP_NAME = selected_candidate.get('crop_name')
    print(json.dumps(selected_candidate, indent=2))
    FORCE_ADAPTER_RESCAN = False

summary = load_adapter_summary(
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
)

CROP_NAME = summary['crop_name']
ADAPTER_DIR = summary['resolved_adapter_dir']
print(f"Resolved crop_name={CROP_NAME}")
print(f"Resolved adapter_dir={ADAPTER_DIR}")

print(json.dumps(summary, indent=2))

In [ ]:
request_new_upload = running_in_colab() and not bool(REUSE_EXISTING_IMAGE_PATH)
adapter_summary = globals().get('summary')
if (ADAPTER_DIR is None or CROP_NAME is None) and isinstance(adapter_summary, dict):
    ADAPTER_DIR = adapter_summary.get('resolved_adapter_dir', ADAPTER_DIR)
    CROP_NAME = adapter_summary.get('crop_name', CROP_NAME)

if ADAPTER_DIR is None and CROP_NAME is None:
    raise RuntimeError(
        'Run the adapter selection/summary cell first so Notebook 3 can resolve ADAPTER_DIR and CROP_NAME before single-image prediction.'
    )

if FORCE_IMAGE_UPLOAD:
    IMAGE_PATH = None
    print('FORCE_IMAGE_UPLOAD=True, requesting a new image upload.')
elif request_new_upload:
    IMAGE_PATH = None
    print('REUSE_EXISTING_IMAGE_PATH=False, requesting a fresh image upload for this run.')

if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Set IMAGE_PATH to one image file when running outside Colab, for example D:/data/leaf.jpg.')
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = next(iter(uploaded.keys()))
    FORCE_IMAGE_UPLOAD = False

IMAGE_PATH = str(Path(IMAGE_PATH))
print(f'Using IMAGE_PATH: {IMAGE_PATH}')

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
)

print(json.dumps(single_result, indent=2))

In [ ]:
if not BATCH_IMAGE_DIR:
    print('Set BATCH_IMAGE_DIR to a folder containing image files to run the optional folder sanity check.')
else:
    rows = predict_image_folder(
        BATCH_IMAGE_DIR,
        CROP_NAME,
        adapter_dir=ADAPTER_DIR,
        adapter_root=ADAPTER_ROOT,
        config_env=CONFIG_ENV,
        device=DEVICE,
    )
    if pd is not None:
        df = pd.DataFrame(rows)
        display(df)
    else:
        print(rows)

    predicted_counts = Counter(row['predicted_class'] for row in rows if row.get('predicted_class'))
    ood_count = sum(1 for row in rows if row.get('is_ood') is True)
    error_count = sum(1 for row in rows if row.get('error'))

    print('predicted_class_counts:', dict(predicted_counts))
    print('ood_count:', ood_count)
    print('error_count:', error_count)